## Feature-Combination

In [27]:
import numpy as np
import csv
from itertools import product
from tqdm import tqdm

# =========================
# CONFIG
# =========================
OUTPUT_PATH = "/Users/zixuanzhao/Desktop/MKG/JMI/Feature-Combination/Cu_Combinations.csv"
CHUNK_SIZE = 5000

# =========================
# 1. 定义变量范围
# =========================
param_space = {
    "Al": {"min": 0.3, "max": 0.5, "step": 0.1},    
    "Cr": {"min": 0.1, "max": 0.3, "step": 0.2},      
    "Mg": {"min": 0.01, "max": 0.16, "step": 0.05},  
    "Ni": {"min": 4.0, "max": 6.0, "step": 0.5},   
    "Si": {"min": 1.0, "max": 2.5, "step": 0.5},       
    "Solution_Temp": {"min": 900, "max": 1000, "step": 50},
    "Solution_Time": {"min": 3, "max": 7, "step": 1},
    "CR_Reduction": {"min": 25, "max": 50, "step": 25},
    "Aging_Temp": {"min": 300, "max": 550, "step": 50},
    "Aging_Time": {"min": 0, "max": 5, "step": 0.5},
    "Processing_Route": {"min": 2, "max": 2, "step": 1}
}

# =========================
# 2. 生成取值
# =========================
value_lists = {}
for k, v in param_space.items():
    value_lists[k] = np.arange(v["min"], v["max"] + v["step"], v["step"])

keys = list(value_lists.keys()) + ["Cu"]

# =========================
# 3. 计算总数
# =========================
total = 1
for v in value_lists.values():
    total *= len(v)

print(f"Total theoretical combinations: {total:,}")

# =========================
# 4. 初始化 CSV
# =========================
with open(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(keys)   # 写表头

buffer = []
count = 0

# =========================
# 5. 主循环（带进度条 + 分批写）
# =========================
for comb in tqdm(product(*value_lists.values()), total=total):
    row = dict(zip(value_lists.keys(), comb))

    # Cu补齐
    alloy_sum = (
        row["Cr"] + row["Ni"] + row["Si"] + row["Al"] + row["Mg"]   
    )
    Cu = 100 - alloy_sum

    # 过滤
    if Cu < 0 or Cu < 80:
        continue

    row["Cu"] = round(Cu, 4)

    buffer.append([row[k] for k in keys])
    count += 1

    # ⭐ 分批写入
    if len(buffer) >= CHUNK_SIZE:
        with open(OUTPUT_PATH, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerows(buffer)

        buffer = []

# 写剩余
if buffer:
    with open(OUTPUT_PATH, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerows(buffer)

print(f"\nSaved: {OUTPUT_PATH}")
print(f"Valid combinations: {count}") 

Total theoretical combinations: 950,400


100%|██████████| 950400/950400 [00:06<00:00, 154109.99it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI/Feature-Combination/Cu_Combinations.csv
Valid combinations: 950400


## 特征转换

In [28]:
"""
特征转换模块
负责从原始输入转换为模型所需的特征
"""

import math
import re
import numpy as np
import pandas as pd
from tqdm import tqdm   # ⭐ 新增

# =========================================================
# 基础常数
# =========================================================
ELEMENTS = ["Cu", "Al", "Cr", "Mg", "Ni", "Si", "Zr"]

ATOMIC_MASS = {
    "Cu": 63.546, "Al": 26.982, "Cr": 51.996,
    "Mg": 24.305, "Ni": 58.693, "Si": 28.085, "Zr": 91.224,
}

ATOMIC_RADIUS = {
    "Cu": 1.28, "Al": 1.43, "Cr": 1.28,
    "Mg": 1.60, "Ni": 1.24, "Si": 1.11, "Zr": 1.60,
}

ELECTRONEGATIVITY = {
    "Cu": 1.90, "Al": 1.61, "Cr": 1.66,
    "Mg": 1.31, "Ni": 1.91, "Si": 1.90, "Zr": 1.33,
}

VEC = {
    "Cu": 11, "Al": 3, "Cr": 6,
    "Mg": 2, "Ni": 10, "Si": 4, "Zr": 4,
}

MELTING_POINT = {
    "Cu": 1084.62, "Al": 660.32, "Cr": 1907.0,
    "Mg": 650.0, "Ni": 1455.0, "Si": 1414.0, "Zr": 1855.0,
}


# =========================================================
# 工具函数
# =========================================================
def extract_number(value) -> float:
    """
    从值中提取数值
    支持:
    - 0.5
    - "0.5"
    - "0.5/wt.%"
    - "980℃"
    - "50%"
    """
    if value is None:
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    s = str(value).strip()
    if s == "" or s.upper() in {"N", "NA", "N/A", "NONE", "NAN"}:
        return np.nan

    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group()) if m else np.nan


def wt_to_mole_fraction(wt: dict) -> dict:
    """
    重量百分比转换为摩尔分数
    """
    moles = {el: wt.get(el, 0.0) / ATOMIC_MASS[el] for el in ELEMENTS}
    total = sum(moles.values())

    if total <= 0:
        return {el: 0.0 for el in ELEMENTS}

    return {el: moles[el] / total for el in ELEMENTS}


# =========================================================
# 成分特征
# =========================================================
def calc_composition_features(xi: dict) -> dict:
    """
    计算成分特征
    """
    feats = {}

    # 平均熔点 Tmavg (℃)
    feats["Tmavg"] = sum(xi[el] * MELTING_POINT[el] for el in ELEMENTS)

    # 平均价电子浓度 VECavg / 方差 VECvar
    feats["VECavg"] = sum(xi[el] * VEC[el] for el in ELEMENTS)
    feats["VECvar"] = sum(xi[el] * (VEC[el] ** 2) for el in ELEMENTS) - (feats["VECavg"] ** 2)

    # 原子尺寸错配 δ
    r_avg = sum(xi[el] * ATOMIC_RADIUS[el] for el in ELEMENTS)
    feats["δ"] = (
        math.sqrt(sum(xi[el] * (1 - ATOMIC_RADIUS[el] / r_avg) ** 2 for el in ELEMENTS))
        if r_avg > 0 else 0.0
    )

    # 电负性方差 χvar
    chi_avg = sum(xi[el] * ELECTRONEGATIVITY[el] for el in ELEMENTS)
    feats["χvar"] = sum(
        xi[el] * (ELECTRONEGATIVITY[el] - chi_avg) ** 2 for el in ELEMENTS
    )

    # 混合熵 Smix (J/mol·K)
    feats["Smix"] = -8.314 * sum(
        xi[el] * math.log(xi[el]) for el in ELEMENTS if xi[el] > 0
    )

    # HHI（成分集中度）
    feats["HHI"] = sum(xi[el] ** 2 for el in ELEMENTS)

    # N_eff（等效参与元素数）
    feats["N_eff"] = 1.0 / feats["HHI"] if feats["HHI"] > 0 else np.nan

    return feats


def classify_family(cr_wt: float, ni_wt: float, si_wt: float) -> int:
    """
    合金体系分类
    0: 非Cr体系 / 非Ni-Si体系
    1: Cr体系
    2: Cr + Ni-Si 复合体系
    """
    has_cr = cr_wt > 0
    has_ni_si = (ni_wt > 0) and (si_wt > 0)

    if has_cr and has_ni_si:
        return 2
    elif has_cr:
        return 1
    else:
        return 0


# =========================================================
# 工艺特征
# =========================================================
def calc_process_features(processing_params: dict, processing_route) -> dict:
    """
    计算工艺特征
    这里 Processing_Route 不做字符串映射，直接保留数值 0/1/2
    """
    feats = {}

    # 提取工艺参数
    s_tem = extract_number(processing_params.get("Solution_Temperature", np.nan))
    a_tem = extract_number(processing_params.get("Aging_Temperature", np.nan))
    a_time = extract_number(processing_params.get("Aging_Time", np.nan))

    cr_red_raw = processing_params.get("CR_Reduction/%", None)
    if cr_red_raw is None:
        cr_red_raw = processing_params.get("Cold_Rolling_Reduction", None)
    if cr_red_raw is None:
        cr_red_raw = processing_params.get("Cold_Rolling", None)
    if cr_red_raw is None:
        cr_red_raw = processing_params.get("CR_Reduction", 0)

    cr_red = extract_number(cr_red_raw)

    # 直接使用数值 route
    try:
        feats["One-Hot-Processing"] = int(float(processing_route))
    except Exception:
        feats["One-Hot-Processing"] = 0

    # 计算工艺衍生特征
    cr_frac = cr_red / 100.0 if not math.isnan(cr_red) else 0.0

    feats["CR_x_ATem"] = cr_frac * a_tem if not math.isnan(a_tem) else np.nan
    feats["ATem_x_Atime"] = a_tem * a_time if not (math.isnan(a_tem) or math.isnan(a_time)) else np.nan
    feats["STem-ATem"] = s_tem - a_tem if not (math.isnan(s_tem) or math.isnan(a_tem)) else np.nan

    return feats


# =========================================================
# 主转换函数
# =========================================================
def transform_input(alloy_composition: dict, processing_params: dict, processing_route) -> pd.DataFrame:
    """
    将原始输入转换为模型所需的特征
    """
    try:
        # -------------------------
        # 1. 输入检查
        # -------------------------
        if not isinstance(alloy_composition, dict):
            alloy_composition = {}

        if not isinstance(processing_params, dict):
            processing_params = {}

        if processing_route is None:
            processing_route = 0

        # -------------------------
        # 2. 提取元素重量百分比
        # -------------------------
        wt = {el: extract_number(alloy_composition.get(el, 0)) for el in ELEMENTS}

        # 缺失值归零
        for el in ELEMENTS:
            if pd.isna(wt[el]):
                wt[el] = 0.0

        # 若没有提供Cu，则自动补齐
        input_has_cu = ("Cu" in alloy_composition) and (
            not pd.isna(extract_number(alloy_composition.get("Cu", np.nan)))
        )
        if not input_has_cu:
            wt["Cu"] = 100.0 - sum(wt[el] for el in ELEMENTS if el != "Cu")

        # 若补齐后 Cu 为负，说明输入不合理
        if wt["Cu"] < 0:
            raise ValueError("Invalid alloy composition: sum of non-Cu elements exceeds 100 wt.%")

        # -------------------------
        # 3. 转换为摩尔分数
        # -------------------------
        xi = wt_to_mole_fraction(wt)

        # -------------------------
        # 4. 计算成分特征
        # -------------------------
        comp_feats = calc_composition_features(xi)

        # -------------------------
        # 5. 计算工艺特征
        # -------------------------
        proc_feats = calc_process_features(processing_params, processing_route)

        # -------------------------
        # 6. 计算其他特征
        # -------------------------
        ni_wt = wt.get("Ni", 0.0)
        si_wt = wt.get("Si", 0.0)
        mg_wt = wt.get("Mg", 0.0)
        cr_wt = wt.get("Cr", 0.0)

        ni_si_wt = ni_wt + si_wt
        family = classify_family(cr_wt, ni_wt, si_wt)

        # Alloying = 基于摩尔分数的非 Cu 含量 (%)
        alloying = (1.0 - xi["Cu"]) * 100.0

        if pd.isna(ni_wt) or pd.isna(si_wt):
            ni_div_si = np.nan
            ni_mul_si = np.nan
        else:
            ni_div_si = (ni_wt / si_wt) if si_wt != 0 else 0.0
            ni_mul_si = ni_wt * si_wt

        # -------------------------
        # 7. 构建特征字典
        # -------------------------
        features = {
            **comp_feats,
            **proc_feats,
            "Si/wt.%": si_wt,
            "Mg/(Ni+Si)": mg_wt / ni_si_wt if ni_si_wt > 0 else 0.0,
            "Ni/Si": ni_div_si,
            "Ni*Si": ni_mul_si,
            "Alloying": alloying,
            "Family": family
        }

        # 处理缺失值
        for key, value in features.items():
            if pd.isna(value) or (isinstance(value, float) and math.isnan(value)):
                features[key] = 0.0 if key in ["Family", "One-Hot-Processing"] else np.nan

        return pd.DataFrame([features])

    except Exception:
        default_features = {
            "HHI": 0.0,
            "N_eff": 0.0,
            "Tmavg": 0.0,
            "δ": 0.0,
            "VECavg": 0.0,
            "VECvar": 0.0,
            "χvar": 0.0,
            "Smix": 0.0,
            "CR_x_ATem": 0.0,
            "ATem_x_Atime": 0.0,
            "STem-ATem": 0.0,
            "One-Hot-Processing": 0,
            "Si/wt.%": 0.0,
            "Mg/(Ni+Si)": 0.0,
            "Ni/Si": 0.0,
            "Ni*Si": 0.0,
            "Alloying": 0.0,
            "Family": 0
        }
        return pd.DataFrame([default_features])


# =========================================================
# 批量转换：把组合表转成特征表
# =========================================================
def batch_transform_csv(
    input_csv_path: str,
    output_csv_path: str,
    composition_cols=None,
    route_col="Processing_Route"
):
    """
    将组合表 CSV 批量转换为特征表 CSV
    Processing_Route 直接保留数值，不再映射字符串
    """

    if composition_cols is None:
        composition_cols = ["Cu", "Al", "Cr", "Mg", "Ni", "Si", "Zr"]

    df_in = pd.read_csv(input_csv_path)

    all_rows = []

    # ⭐ 总行数
    total = len(df_in)

    # =========================
    # ⭐ 核心进度条（这里）
    # =========================
    for row in tqdm(df_in.itertuples(index=False), total=total, desc="Feature Engineering"):

        row_dict = row._asdict()

        # 1) 成分
        alloy_composition = {
            col: row_dict.get(col, 0.0) for col in composition_cols
        }

        # 2) 工艺参数
        processing_params = {
            "Solution_Temperature": row_dict.get("Solution_Temp", np.nan),
            "Aging_Temperature": row_dict.get("Aging_Temp", np.nan),
            "Aging_Time": row_dict.get("Aging_Time", np.nan),
            "CR_Reduction": row_dict.get("CR_Reduction", 0.0),
        }

        # 3) route
        processing_route = row_dict.get(route_col, 0)

        # 4) 特征转换
        feat_df = transform_input(
            alloy_composition=alloy_composition,
            processing_params=processing_params,
            processing_route=processing_route
        )

        feat_row = feat_df.iloc[0].to_dict()

        # 5) 合并
        merged_row = {
            **row_dict,
            **feat_row
        }

        all_rows.append(merged_row)

    # =========================
    # 输出
    # =========================
    df_out = pd.DataFrame(all_rows)
    df_out.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"\nSaved to: {output_csv_path}")
    print(f"Total rows: {len(df_out)}")

    return df_out


# =========================================================
# 示例
# =========================================================
if __name__ == "__main__":
    # 单条输入示例
    alloy_composition = {
        "Cu": 97.0,
        "Ni": 2.48,
        "Si": 0.52
    }

    processing_params = {
        "Solution_Temperature": 980,
        "Aging_Temperature": 450,
        "Aging_Time": 5,
        "CR_Reduction": 50
    }

    processing_route = 1   # 直接保留数值

    df_feat = transform_input(alloy_composition, processing_params, processing_route)
    print(df_feat)

    # 批量示例（按需取消注释）
    df_out = batch_transform_csv(
        input_csv_path="/Users/zixuanzhao/Desktop/MKG/JMI/Feature-Combination/Cu_Combinations.csv",
        output_csv_path="/Users/zixuanzhao/Desktop/MKG/JMI/Feature-Combination/Cu_Combinations_Features.csv"
    )
    print(df_out.head()) 

         Tmavg     VECavg    VECvar         δ      χvar      Smix       HHI  \
0  1098.322235  10.891723  0.586491  0.015072  0.000003  1.546379  0.925738   

      N_eff  One-Hot-Processing  CR_x_ATem  ATem_x_Atime  STem-ATem  Si/wt.%  \
0  1.080219                   1      225.0        2250.0      530.0     0.52   

   Mg/(Ni+Si)     Ni/Si   Ni*Si  Alloying  Family  
0         0.0  4.769231  1.2896  3.828637       0  


Feature Engineering: 100%|██████████| 950400/950400 [03:13<00:00, 4922.66it/s]



Saved to: /Users/zixuanzhao/Desktop/MKG/JMI/Feature-Combination/Cu_Combinations_Features.csv
Total rows: 950400
    Al   Cr    Mg   Ni   Si  Solution_Temp  Solution_Time  CR_Reduction  \
0  0.3  0.1  0.01  4.0  1.0            900              3            25   
1  0.3  0.1  0.01  4.0  1.0            900              3            25   
2  0.3  0.1  0.01  4.0  1.0            900              3            25   
3  0.3  0.1  0.01  4.0  1.0            900              3            25   
4  0.3  0.1  0.01  4.0  1.0            900              3            25   

   Aging_Temp  Aging_Time  ...  One-Hot-Processing  CR_x_ATem  ATem_x_Atime  \
0         300         0.0  ...                 2.0       75.0           0.0   
1         300         0.5  ...                 2.0       75.0         150.0   
2         300         1.0  ...                 2.0       75.0         300.0   
3         300         1.5  ...                 2.0       75.0         450.0   
4         300         2.0  ...           

## 模型

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

import lightgbm as lgb
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# =========================================================
# PATH
# =========================================================
TRAIN_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI/Feature.xlsx")
PRED_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI/Cu_Combinations1_Features.csv")
OUT_PATH   = Path("/Users/zixuanzhao/Desktop/MKG/JMI/Prediction_Result.xlsx")

# =========================================================
# 模型配置（你给定的）
# =========================================================

CONFIG = {
    "HV": {
        "target": "Hardness/HV",
        "model": "LGBM",
        "features": ['ATem_x_Atime', 'STem-ATem', 'CR_x_ATem', 'Tmavg', 'Family', 'VECvar'],
        "params": {
            'learning_rate': 0.05375154580798958,
            'num_leaves': 85,
            'max_depth': 7,
            'min_child_samples': 93,
            'subsample': 0.8735756585963746,
            'colsample_bytree': 0.9167055831848931,
            'reg_alpha': 0.09197450544145801,
            'reg_lambda': 0.06908205918528182,
            'min_split_gain': 0.1070946938483279,
            'n_estimators': 800,
            'random_state': 42,
            'n_jobs': -1,
            'verbosity': -1
        }
    },

    "EC": {
        "target": "EC/%IACS",
        "model": "XGB",
        "features": ['ATem_x_Atime', 'STem-ATem', 'CR_x_ATem',
                     'Ni/Si', 'Ni*Si', 'N_eff', 'Tmavg', 'Alloying', 'VECavg'],
        "params": {
            'learning_rate': 0.07936119920354771,
            'max_depth': 6,
            'min_child_weight': 14.422782131926159,
            'subsample': 0.9959679705334656,
            'colsample_bytree': 0.7977094996436664,
            'gamma': 4.085613992186858,
            'reg_alpha': 0.6164395811668538,
            'reg_lambda': 42.47048763033667,
            'n_estimators': 800,
            'tree_method': 'hist',
            'random_state': 42,
            'n_jobs': -1
        }
    },

    "Q3": {
        "target": "Q3-Euclidean",
        "model": "XGB",
        "features": ['STem-ATem', 'One-Hot-Processing', 'CR_x_ATem',
                     'Mg/(Ni+Si)', 'Tmavg', 'χvar', 'Smix', 'δ', 'VECvar'],
        "params": {
            'learning_rate': 0.028013565282778832,
            'max_depth': 8,
            'min_child_weight': 12.089756742999135,
            'subsample': 0.827605319863244,
            'colsample_bytree': 0.7316649093096297,
            'gamma': 0.007004273033412803,
            'reg_alpha': 0.00017271709120985208,
            'reg_lambda': 0.28890545401315165,
            'n_estimators': 800,
            'tree_method': 'hist',
            'random_state': 42,
            'n_jobs': -1
        }
    }
}

# =========================================================
# 预处理
# =========================================================
def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", oh)
        ]), cat_cols)
    ])

# =========================================================
# 主流程
# =========================================================
def main():

    train_df = pd.read_excel(TRAIN_PATH, sheet_name="Feature")
    pred_df  = pd.read_csv(PRED_PATH)

    out_df = pred_df.copy()

    for name, cfg in CONFIG.items():

        print(f"\n=== Running {name} ===")

        feats = cfg["features"]
        target = cfg["target"]

        # ===== 训练集 =====
        train_sub = train_df[feats + [target]].dropna().copy()

        X_train = train_sub[feats]
        y_train = train_sub[target].astype(float)

        # ===== 预测集 =====
        X_pred = pred_df[feats].copy()

        # ===== 预处理 =====
        pre = build_preprocessor(X_train)
        Z_train = pre.fit_transform(X_train)
        Z_pred  = pre.transform(X_pred)

        # ===== 模型 =====
        if cfg["model"] == "LGBM":
            model = lgb.LGBMRegressor(**cfg["params"])
        else:
            model = xgb.XGBRegressor(**cfg["params"])

        model.fit(Z_train, y_train)

        y_pred = model.predict(Z_pred)

        out_df[f"Pred_{target}"] = y_pred

    # ===== 额外指标 =====
    if "Pred_Hardness/HV" in out_df.columns and "Pred_EC/%IACS" in out_df.columns:
        out_df["Pred_HVxEC"] = out_df["Pred_Hardness/HV"] * out_df["Pred_EC/%IACS"]

    # ===== 保存 =====
    out_df.to_excel(OUT_PATH, index=False)

    print("\nSaved:", OUT_PATH)
    print(out_df.head())

if __name__ == "__main__":
    main()

## 合并

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# PATH
# =========================
INPUT_PATH = Path("/mnt/data/Performance.xlsx")
OUTPUT_PATH = Path("/mnt/data/Performance_Final_Merged.xlsx")

# =========================
# 成分列（按这些合并）
# =========================
COMP_COLS = ["Cu", "Cr", "Ni", "Si"]

# =========================
# 所有需要压缩的列（🔥已包含你新增的）
# =========================
RANGE_COLS = [
    "Solution_Temp",
    "Solution_Time",
    "CR_Reduction",
    "Aging_Temp",
    "Aging_Time",

    "Pred_Hardness/HV",
    "Pred_EC/%IACS",
    "Pred_Q3-Euclidean",
    "HV*EC",
]

ROUTE_COL = "Processing_Route"


# =========================
# 判断整数
# =========================
def is_int_like(x):
    return abs(float(x) - round(float(x))) < 1e-8


# =========================
# 估计步长（核心）
# =========================
def estimate_step(vals):
    if len(vals) < 2:
        return None

    diffs = np.diff(vals)
    diffs = diffs[np.abs(diffs) > 1e-8]

    if len(diffs) == 0:
        return None

    step = np.min(np.abs(diffs))

    # 是否基本等步长
    if np.all(np.abs(diffs / step - np.round(diffs / step)) < 1e-6):
        return step

    return None


# =========================
# 区间压缩（通用）
# =========================
def compress_ranges(values):
    vals = pd.Series(values).dropna().tolist()
    if len(vals) == 0:
        return ""

    vals = sorted(set(round(float(v), 6) for v in vals))

    if len(vals) == 1:
        return str(int(vals[0])) if is_int_like(vals[0]) else str(vals[0])

    step = estimate_step(np.array(vals))

    # 不规则 → min-max
    if step is None:
        return f"{vals[0]}-{vals[-1]}"

    ranges = []
    start = vals[0]
    prev = vals[0]

    for v in vals[1:]:
        if abs((v - prev) - step) < 1e-6:
            prev = v
        else:
            ranges.append((start, prev))
            start = v
            prev = v

    ranges.append((start, prev))

    def fmt(x):
        return str(int(x)) if is_int_like(x) else str(round(x, 6))

    result = []
    for a, b in ranges:
        if abs(a - b) < 1e-8:
            result.append(fmt(a))
        else:
            result.append(f"{fmt(a)}-{fmt(b)}")

    return ",".join(result)


# =========================
# Processing Route 专用
# =========================
def compress_route(values):
    vals = sorted(set(pd.Series(values).dropna().tolist()))
    return ",".join(map(str, vals))


# =========================
# 主流程
# =========================
def main():
    df = pd.read_excel(INPUT_PATH)

    # 转数值
    for c in RANGE_COLS + COMP_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    rows = []

    grouped = df.groupby(COMP_COLS, dropna=False)

    for keys, sub in grouped:
        row = {}

        # 成分
        for col, val in zip(COMP_COLS, keys):
            row[col] = val

        # 数值区间列
        for c in RANGE_COLS:
            row[c] = compress_ranges(sub[c].tolist())

        # 工艺路线（特殊）
        if ROUTE_COL in sub.columns:
            row[ROUTE_COL] = compress_route(sub[ROUTE_COL])

        # 数量
        row["Count"] = len(sub)

        rows.append(row)

    out_df = pd.DataFrame(rows)

    # 列顺序
    ordered_cols = (
        COMP_COLS +
        [ROUTE_COL] +
        RANGE_COLS +
        ["Count"]
    )

    ordered_cols = [c for c in ordered_cols if c in out_df.columns]
    other_cols = [c for c in out_df.columns if c not in ordered_cols]

    out_df = out_df[ordered_cols + other_cols]

    out_df.to_excel(OUTPUT_PATH, index=False)

    print("Saved:", OUTPUT_PATH)
    print(out_df.head())


if __name__ == "__main__":
    main()